<a href="https://colab.research.google.com/github/Abhinaytechie/LangGraph/blob/main/memoryschema_profile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install  langchain_groq langgraph langchain_core trustcall

In [ ]:
from langgraph.store.memory import InMemoryStore
from langchain_groq import ChatGroq
from google.colab import userdata
from pydantic import BaseModel,Field
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage
api_key=userdata.get('GROQ_API_KEY')
llm=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=api_key)

In [ ]:
in_memory_store = InMemoryStore()

class UserProfile(BaseModel):
  "Extract user profile information" # Added docstring for description
  content:str=Field("info about the user")
class Profiles(BaseModel):
  "segmented user info" # Added docstring for description
  profiles:list[UserProfile]=Field(description="List of user profiles")
structured_llm=llm.with_structured_output(Profiles)

In [ ]:
structured_llm.invoke([HumanMessage(content="I am Abhinay,i am proffiecnt in python,java,langchain,langgraph i have experience as full-stackdevloper at Spotmies Llp,python developer in viswam")]).profiles

[UserProfile(content='Proficient in Python, Java, LangChain, LangGraph'),
 UserProfile(content='Experience as Full-Stack Developer at Spotmies LLP'),
 UserProfile(content='Experience as Python Developer at Viswam')]

In [ ]:
structured_llm.invoke([HumanMessage(content="I am Abhinay,i am proffiecnt in langraph also")]).profiles


[UserProfile(name='Abhinay', skills=['Python,Langchain'], internships='Full-Stack Developer at Spotmies LLP')]

In [ ]:
# Conversation
conversation = [HumanMessage(content="Hi, I'm Abhinay."),
                AIMessage(content="Nice to meet you, Abhinay."),
                HumanMessage(content="I really like building functional apps with langchain and fastapi")]

In [ ]:
from trustcall import create_extractor

extractor=create_extractor(
    llm,
    tools=[UserProfile],
    tool_choice="UserProfile",
    enable_inserts=True
)
system_msg="Take the conversation and Get the profile info from the given prompt"

res=extractor.invoke({"messages":[SystemMessage(content=system_msg)]+conversation})


In [ ]:
from pprint import pprint
for m in res['messages']:
  m.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  UserProfile (aggnr17jc)
 Call ID: aggnr17jc
  Args:
    content: Abhinay is a developer who enjoys building functional apps using langchain and fastapi
    name: Abhinay


In [ ]:
schema = res["responses"]
schema


[UserProfile(name='Abhinay', skills=['langchain', 'fastapi', 'functional app development'], internships='none')]

In [ ]:
metadata=res['response_metadata']
metadata

[{'id': '8ba2kfv5t'}]

In [ ]:
schema[0].model_dump()

{'name': 'Abhinay',
 'skills': ['building functional apps', 'langchain', 'fastapi'],
 'internships': []}

In [ ]:
updated_conversation=[
    HumanMessage(content="Hi, I'm Abhinay."),
    AIMessage(content="Nice to meet you, Abhinay."),
    HumanMessage(content="I really like building functional apps with langchain and fastapi."),
    AIMessage(content="That's awesome! LangChain and FastAPI make a powerful combo for AI-driven apps."),
    HumanMessage(content="Yeah, I recently built an NLP-to-SQL agent using them."),
    AIMessage(content="Wow, that sounds impressive! Did you also integrate any database for the queries?"),
    HumanMessage(content="Yes, I connected it to a PostgreSQL database to execute real-time queries."),
    AIMessage(content="Nice! You're definitely getting hands-on with production-level AI systems."),
    HumanMessage(content="I know about MongoDB also."),
    AIMessage(content="Nice! You are impressive with it ryt?.")
]


In [ ]:
namespace=("memory","1")
tool_name="UserProfile"
exist_mem=[(str(i),tool_name,profile.model_dump()) for i,profile in enumerate(res['responses'])]if res['responses'] else None
exist_mem

[('0',
  'UserProfile',
  {'content': 'Abhinay is a developer who enjoys building functional apps using langchain and fastapi'})]

In [ ]:
# Invoke the extractor with our updated conversation and existing memories
result = extractor.invoke({"messages": [SystemMessage(content="See the profiles in the collection and update or create new profiles based on the followng conversation")]+updated_conversation,
                                     "existing": exist_mem})

In [ ]:
for m in result['messages']:
  m.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  UserProfile (26pg7ktr7)
 Call ID: 26pg7ktr7
  Args:
    content: Abhinay is a developer who enjoys building functional apps using langchain and fastapi, has experience with postgresql and mongodb, and recently built an NLP-to-SQL agent


In [ ]:
system_msg="Update the memory (JSON doc ) from the following conversation."
res=extractor.invoke({"messages":[SystemMessage(content=system_msg)]+conversation},
                 {'existing':schema[0].model_dump()})
print(res['responses'])

[UserProfile(name='Abhinay', skills=['LangChain', 'FastAPI', 'PostgreSQL', 'NLP', 'SQL'], internships=['NLP-to-SQL Agent'])]


In [ ]:
from pydantic import BaseModel, Field

class Memory(BaseModel):
    content: str = Field(description="The main content of the memory. For example: User expressed interest in learning about French.")

class MemoryCollection(BaseModel):
    memories: list[Memory] = Field(description="A list of memories about the user.")